# Diagnose Perfect-Correction Slope Excess

This CPU characterization notebook tests whether the excess entanglement-slope signal
$$\delta m_s = m_s - \frac{1}{3}$$
in the trajectory-resolved `perfect_correction` data is primarily explained by global charge imbalance, local wall charge/defect structure, incomplete cycle convergence, averaging order, or localized contour contamination.

It uses saved data only. It does not run adaptive dynamics or mutate source data.

## 1. Setup and Data Manifest

The main data source is the saved N20 streaming-observable campaign for `Ny = 30, 40, 50`. Optional raw-covariance diagnostics are gated off by default because they are heavier and use the N16 snapshot products only as a secondary check.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from scipy.optimize import curve_fit
from scipy.stats import pearsonr, spearmanr

# Keep matplotlib cache writes out of the repository when possible.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_slope_excess")

ROOT = Path.cwd().resolve()
if not (ROOT / "REPO_POLICY.md").exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "REPO_POLICY.md").exists():
            ROOT = parent
            break
assert (ROOT / "REPO_POLICY.md").exists(), f"Could not locate repo root from {Path.cwd()}"

BUNDLE = ROOT / "colab_charge_fluctuations"
CAMPAIGN_ID = "N20_selected_nsh1_dwtrunc1_init-default_S100_cycles-2Ny"
OBS_ROOT = BUNDLE / "gpu_data/streaming_covariance_observables/campaigns" / CAMPAIGN_ID / "runs"
CHAR_ROOT = BUNDLE / "analysis_outputs/streaming_covariance_characterization" / CAMPAIGN_ID / "tables"
OUTPUT_ROOT = BUNDLE / "analysis_outputs/perfect_correction_slope_excess_diagnostics"
FIG_ROOT = OUTPUT_ROOT / "figures"
TABLE_ROOT = OUTPUT_ROOT / "tables"
for path in (OUTPUT_ROOT, FIG_ROOT, TABLE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

GEOMETRIES_NY = [30, 40, 50]
PROTOCOL = "perfect_correction"
TARGET_SLOPE = 1.0 / 3.0
FIT_AY_MIN = 8
WALL_RADIUS = 2
RETAINED_FRACTIONS = [0.10, 0.20, 0.30, 0.50, 1.00]
RUN_RAW_COVARIANCE_SECTION = False
RAW_SAMPLE_LIMIT = 4
RAW_Y0_STRIDE = 4
RAW_NY = 30
RAW_NSHELL = 1
SAVE_FIGURES = True
SAVE_TABLES = True
DPI = 300
FIG_WIDTH = 3.375

mpl.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "font.family": "sans-serif",
    "font.sans-serif": ["CMU Sans Serif", "DejaVu Sans", "Arial"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

NY_COLORS = {30: "#1f77b4", 40: "#2ca02c", 50: "#d62728"}
METRIC_LABELS = {
    "abs_q_pct": r"$|\Delta Q|$ (%)",
    "charge_spatial_rms": r"charge RMS",
    "wall_charge_rms": r"wall charge RMS",
    "bulk_charge_rms": r"bulk charge RMS",
    "wall_frustration_mean": r"wall frustration",
    "bulk_frustration_mean": r"bulk frustration",
    "frustration_full_mean": r"mean frustration",
    "chern_error": r"$|C-1|$",
    "frob_successive_delta": r"$\|G_t-G_{t-1}\|_F$",
    "entropy_slope": r"$m_s$",
    "delta_slope": r"$m_s-1/3$",
}

def ensure_no_forbidden_paths(paths):
    forbidden_dir = "erroneous" + "_gpu_stuff"
    bad = [str(p) for p in paths if forbidden_dir in Path(p).parts]
    if bad:
        raise RuntimeError(f"Forbidden {forbidden_dir} paths were selected:\n" + "\n".join(bad))

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_ROOT / name
    if SAVE_TABLES:
        df.to_csv(path, index=False)
    return path

def save_figure(fig: plt.Figure, stem: str) -> None:
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIG_ROOT / f"{stem}.pdf", bbox_inches="tight")
        fig.savefig(FIG_ROOT / f"{stem}.png", bbox_inches="tight")

def sem(values) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size <= 1:
        return np.nan
    return float(np.std(arr, ddof=1) / np.sqrt(arr.size))

def log_chord(ay, ny: int) -> np.ndarray:
    ay = np.asarray(ay, dtype=float)
    return np.log(np.sin(np.pi * ay / float(ny)))

def linear_fit_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 2 or np.allclose(x, x[0]):
        return np.nan, np.nan, np.nan, int(x.size)
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    sse = float(np.sum((y - pred) ** 2))
    sst = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - sse / sst if sst > 0 else np.nan
    return float(slope), float(intercept), float(r2), int(x.size)

def fit_entropy_curve(ay_values, entropy_curve, ny: int, ay_min: int = FIT_AY_MIN):
    ay = np.asarray(ay_values, dtype=int)
    curve = np.asarray(entropy_curve, dtype=float)
    mask = (ay >= int(ay_min)) & (ay <= int(ny) // 2)
    return linear_fit_1d(log_chord(ay[mask], ny), curve[mask])

def domain_wall_masks(nx: int, radius: int = WALL_RADIUS):
    half = nx // 2
    w = nx // 4
    xL = half - w
    xR = half + w
    x = np.arange(nx)
    wall = (np.abs(x - xL) <= radius) | (np.abs(x - xR) <= radius)
    bulk = ~wall
    return xL, xR, wall, bulk

forbidden_call = "run_" + "markov_circuit("
assert forbidden_call not in "".join(open(__file__).read() if "__file__" in globals() else ""), "unexpected dynamics call literal"
print(f"Repo root: {ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

In [ ]:
# Load the small manifest and source products. This cell intentionally reads saved observables only.
source_paths = []
scatter_path = CHAR_ROOT / "perfect_scatter_cycles_ge20.csv"
assert scatter_path.exists(), scatter_path
source_paths.append(scatter_path)
scatter = pd.read_csv(scatter_path)
scatter = scatter[(scatter["protocol"] == PROTOCOL) & (scatter["Ny"].isin(GEOMETRIES_NY))].copy()
scatter["delta_slope"] = scatter["entropy_slope"] - TARGET_SLOPE
scatter["abs_q_pct"] = scatter["q_pct"].abs()

RUN_DATA = {}
manifest_rows = []
for ny in GEOMETRIES_NY:
    run_dir = OBS_ROOT / f"N20x{ny}_nsh1_init-default_{PROTOCOL}"
    paths = {
        "latest": run_dir / "latest_run.json",
        "scalar": run_dir / "scalar_metrics.csv",
        "local_charge": run_dir / "local_charge_cell.npz",
        "entropy": run_dir / "entropy_y0avg_vs_ay.npz",
        "frustration": run_dir / "measurement_frustration.npz",
    }
    missing = [name for name, path in paths.items() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing {missing} for Ny={ny} in {run_dir}")
    source_paths.extend(paths.values())

    with paths["latest"].open("r", encoding="utf-8") as fh:
        latest = json.load(fh)
    scalar = pd.read_csv(paths["scalar"])
    local_npz = np.load(paths["local_charge"], allow_pickle=False)
    entropy_npz = np.load(paths["entropy"], allow_pickle=False)
    frustration_npz = np.load(paths["frustration"], allow_pickle=False)

    local_cycle = local_npz["cycle_labels"].astype(int)
    local_sample = local_npz["sample_indices"].astype(int)
    ent_cycle = entropy_npz["cycle_labels"].astype(int)
    ent_sample = entropy_npz["sample_indices"].astype(int)
    fr_cycle = frustration_npz["cycle_labels"].astype(int)
    fr_sample = frustration_npz["sample_indices"].astype(int)
    assert np.array_equal(local_cycle, ent_cycle) and np.array_equal(local_cycle, fr_cycle)
    assert np.array_equal(local_sample, ent_sample) and np.array_equal(local_sample, fr_sample)

    RUN_DATA[ny] = {
        "run_dir": run_dir,
        "latest": latest,
        "scalar": scalar,
        "local_npz": local_npz,
        "entropy_npz": entropy_npz,
        "frustration_npz": frustration_npz,
        "cycle_labels": local_cycle,
        "sample_indices": local_sample,
    }
    manifest_rows.append({
        "Ny": ny,
        "run_dir": str(run_dir.relative_to(ROOT)),
        "samples": int(local_sample.size),
        "cycles": int(local_cycle.size),
        "cycle_min": int(local_cycle.min()),
        "cycle_max": int(local_cycle.max()),
        "local_charge_shape": tuple(local_npz["local_charge_cell"].shape),
        "frustration_shape": tuple(frustration_npz["frustration_avg"].shape),
        "entropy_shape": tuple(entropy_npz["entropy_y0avg_vs_ay"].shape),
        "source": "saved_observable_products_only",
    })

ensure_no_forbidden_paths(source_paths)
manifest = pd.DataFrame(manifest_rows)
save_table(manifest, "data_manifest.csv")
display(manifest)
print(f"Loaded scatter rows: {len(scatter):,}")

## 2. Feature Table Construction

One row corresponds to one trajectory at one observed cycle, indexed by `(Ny, sample_index, cycle_label)`. The wall masks use the current convention `half = Nx//2`, `w = Nx//4`, and walls at `xL = half-w`, `xR = half+w` with a configurable integer radius.

In [ ]:
def observable_feature_frame_for_ny(ny: int) -> pd.DataFrame:
    data = RUN_DATA[ny]
    local = data["local_npz"]["local_charge_cell"].astype(np.float64, copy=False)
    frust = data["frustration_npz"]["frustration_avg"].astype(np.float64, copy=False)
    sample_indices = data["sample_indices"]
    cycle_labels = data["cycle_labels"]
    nx = int(local.shape[2])
    xL, xR, wall_mask, bulk_mask = domain_wall_masks(nx, WALL_RADIUS)

    mean_charge = local.mean(axis=(2, 3))
    centered = local - mean_charge[:, :, None, None]
    charge_spatial_rms = np.sqrt(np.mean(centered * centered, axis=(2, 3)))
    wall_charge = local[:, :, wall_mask, :]
    bulk_charge = local[:, :, bulk_mask, :]
    wall_charge_rms = np.sqrt(np.mean(wall_charge * wall_charge, axis=(2, 3)))
    bulk_charge_rms = np.sqrt(np.mean(bulk_charge * bulk_charge, axis=(2, 3)))
    wall_charge_abs_mean = np.mean(np.abs(wall_charge), axis=(2, 3))
    bulk_charge_abs_mean = np.mean(np.abs(bulk_charge), axis=(2, 3))

    frust_full = np.mean(frust, axis=(2, 3))
    frust_wall = np.mean(frust[:, :, wall_mask, :], axis=(2, 3))
    frust_bulk = np.mean(frust[:, :, bulk_mask, :], axis=(2, 3))
    frust_rms = np.sqrt(np.mean(frust * frust, axis=(2, 3)))

    rows = []
    for si, sample_index in enumerate(sample_indices):
        for ci, cycle_label in enumerate(cycle_labels):
            rows.append({
                "Ny": int(ny),
                "Nx": nx,
                "sample_index": int(sample_index),
                "cycle_label": int(cycle_label),
                "charge_spatial_mean": float(mean_charge[si, ci]),
                "charge_spatial_rms": float(charge_spatial_rms[si, ci]),
                "wall_charge_rms": float(wall_charge_rms[si, ci]),
                "bulk_charge_rms": float(bulk_charge_rms[si, ci]),
                "wall_charge_abs_mean": float(wall_charge_abs_mean[si, ci]),
                "bulk_charge_abs_mean": float(bulk_charge_abs_mean[si, ci]),
                "frustration_full_mean": float(frust_full[si, ci]),
                "wall_frustration_mean": float(frust_wall[si, ci]),
                "bulk_frustration_mean": float(frust_bulk[si, ci]),
                "frustration_full_rms": float(frust_rms[si, ci]),
                "xL": int(xL),
                "xR": int(xR),
                "wall_radius": int(WALL_RADIUS),
                "wall_site_fraction": float(np.mean(wall_mask)),
            })
    return pd.DataFrame(rows)

obs_features = pd.concat([observable_feature_frame_for_ny(ny) for ny in GEOMETRIES_NY], ignore_index=True)
scalar = pd.concat([RUN_DATA[ny]["scalar"] for ny in GEOMETRIES_NY], ignore_index=True)
scalar = scalar[["Ny", "Nx", "sample_index", "cycle_label", "frob_norm", "frob_successive_delta", "real_space_chern"]].copy()
scalar["chern_error"] = (scalar["real_space_chern"] - 1.0).abs()

keys = ["Ny", "Nx", "sample_index", "cycle_label"]
features = scatter.merge(scalar, on=keys, how="left", validate="one_to_one")
features = features.merge(obs_features, on=keys, how="left", validate="one_to_one")
if features.isna().any().any():
    na_cols = features.columns[features.isna().any()].tolist()
    print("Columns with missing values:", na_cols)
    display(features[features[na_cols].isna().any(axis=1)].head())
    raise ValueError("Feature table contains missing values after alignment.")

feature_path = save_table(features, "slope_excess_feature_table.csv")
print(f"Feature table rows: {len(features):,}")
print(f"Saved: {feature_path}")
display(features.head())

In [ ]:
final_cycle_by_ny = features.groupby("Ny")["cycle_label"].max().to_dict()
final_features = pd.concat([
    features[(features["Ny"] == ny) & (features["cycle_label"] == cyc)]
    for ny, cyc in final_cycle_by_ny.items()
], ignore_index=True)

final_summary = (
    final_features.groupby("Ny")
    .agg(
        final_cycle=("cycle_label", "first"),
        n=("entropy_slope", "size"),
        mean_slope=("entropy_slope", "mean"),
        sem_slope=("entropy_slope", sem),
        mean_delta=("delta_slope", "mean"),
        mean_abs_q_pct=("abs_q_pct", "mean"),
        mean_wall_charge_rms=("wall_charge_rms", "mean"),
        mean_wall_frustration=("wall_frustration_mean", "mean"),
        mean_chern_error=("chern_error", "mean"),
        mean_frob_delta=("frob_successive_delta", "mean"),
    )
    .reset_index()
)
save_table(final_summary, "final_cycle_summary.csv")
display(final_summary)

## 3. Test 1: Per-Trajectory Correlations

The first test asks whether trajectories with larger excess slope also have larger global charge imbalance, stronger local charge texture, stronger wall frustration, Chern-marker error, or larger successive covariance motion.

In [ ]:
# Plot controls
SCATTER_METRICS = [
    "abs_q_pct",
    "charge_spatial_rms",
    "wall_charge_rms",
    "wall_frustration_mean",
    "chern_error",
    "frob_successive_delta",
]
SCATTER_TITLE = r"Final-cycle trajectory diagnostics for $m_s-1/3$"
SCATTER_ALPHA = 0.72
SCATTER_SIZE = 12
SCATTER_XSCALE = "linear"
SCATTER_YSCALE = "linear"
SCATTER_YLIM = None
SCATTER_LEGEND_LOC = "best"
SCATTER_SAVE_STEM = "test1_delta_slope_scatter_matrix"

fig, axes = plt.subplots(2, 3, figsize=(2.10 * FIG_WIDTH, 1.35 * FIG_WIDTH), sharey=True)
axes = axes.ravel()
for ax, metric in zip(axes, SCATTER_METRICS):
    for ny, sub in final_features.groupby("Ny"):
        ax.scatter(
            sub[metric], sub["delta_slope"],
            s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
            color=NY_COLORS.get(int(ny)), label=f"Ny={ny}", linewidths=0,
        )
    ax.axhline(0.0, color="0.2", lw=0.8, ls="--")
    ax.set_xlabel(METRIC_LABELS.get(metric, metric))
    ax.set_xscale(SCATTER_XSCALE)
    ax.set_yscale(SCATTER_YSCALE)
    if SCATTER_YLIM is not None:
        ax.set_ylim(*SCATTER_YLIM)
axes[0].set_ylabel(METRIC_LABELS["delta_slope"])
axes[3].set_ylabel(METRIC_LABELS["delta_slope"])
axes[0].legend(loc=SCATTER_LEGEND_LOC, frameon=False)
fig.suptitle(SCATTER_TITLE, y=1.02, fontsize=9)
save_figure(fig, SCATTER_SAVE_STEM)
plt.show()

In [ ]:
correlation_rows = []
for group_name, sub in [("pooled", final_features), *[(f"Ny={ny}", s) for ny, s in final_features.groupby("Ny")]]:
    for metric in SCATTER_METRICS:
        x = sub[metric].to_numpy(dtype=float)
        y = sub["delta_slope"].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() >= 3 and np.nanstd(x[mask]) > 0 and np.nanstd(y[mask]) > 0:
            pr, pp = pearsonr(x[mask], y[mask])
            sr, sp = spearmanr(x[mask], y[mask])
        else:
            pr = pp = sr = sp = np.nan
        correlation_rows.append({
            "group": group_name,
            "metric": metric,
            "n": int(mask.sum()),
            "pearson_r": float(pr),
            "pearson_p": float(pp),
            "spearman_r": float(sr),
            "spearman_p": float(sp),
        })
correlations = pd.DataFrame(correlation_rows)
save_table(correlations, "test1_metric_correlations.csv")
display(correlations.sort_values(["group", "spearman_r"], ascending=[True, False]))

## 4. Test 2: Conditioned Trajectory Subsets

For each candidate defect metric, keep only the lowest quantiles and ask whether the retained trajectories move toward the target slope $1/3$.

In [ ]:
CONDITION_METRICS = ["abs_q_pct", "charge_spatial_rms", "wall_charge_rms", "wall_frustration_mean", "chern_error"]
condition_rows = []
for ny, sub_ny in final_features.groupby("Ny"):
    for metric in CONDITION_METRICS:
        ordered = sub_ny.sort_values(metric, kind="mergesort")
        total = len(ordered)
        for frac in RETAINED_FRACTIONS:
            n_keep = max(1, int(math.ceil(frac * total)))
            kept = ordered.head(n_keep)
            condition_rows.append({
                "Ny": int(ny),
                "metric": metric,
                "retained_fraction": float(frac),
                "n": int(n_keep),
                "cut_value": float(kept[metric].max()),
                "mean_slope": float(kept["entropy_slope"].mean()),
                "sem_slope": sem(kept["entropy_slope"]),
                "mean_delta": float(kept["delta_slope"].mean()),
                "excess_reduction_fraction": float(
                    1.0 - kept["delta_slope"].mean() / sub_ny["delta_slope"].mean()
                ) if sub_ny["delta_slope"].mean() != 0 else np.nan,
            })
conditioned = pd.DataFrame(condition_rows)
save_table(conditioned, "test2_conditioned_subset_slopes.csv")
display(conditioned.head(15))

In [ ]:
# Plot controls
CONDITION_TITLE = r"Low-defect conditioning of final-cycle entropy slope"
CONDITION_XLABEL = "retained fraction"
CONDITION_YLABEL = r"conditioned mean $m_s$"
CONDITION_YLIM = None
CONDITION_LEGEND_LOC = "best"
CONDITION_MARKER = "o"
CONDITION_SAVE_STEM = "test2_conditioned_subset_slopes"

fig, axes = plt.subplots(2, 3, figsize=(2.1 * FIG_WIDTH, 1.38 * FIG_WIDTH), sharex=True, sharey=True)
axes = axes.ravel()
for ax, metric in zip(axes, CONDITION_METRICS):
    for ny, sub in conditioned[conditioned["metric"] == metric].groupby("Ny"):
        ax.errorbar(
            sub["retained_fraction"], sub["mean_slope"], yerr=sub["sem_slope"],
            marker=CONDITION_MARKER, color=NY_COLORS.get(int(ny)), lw=1.1,
            capsize=2, label=f"Ny={ny}",
        )
    ax.axhline(TARGET_SLOPE, color="0.2", ls="--", lw=0.8)
    ax.set_title(METRIC_LABELS.get(metric, metric))
    ax.set_xlabel(CONDITION_XLABEL)
    if CONDITION_YLIM is not None:
        ax.set_ylim(*CONDITION_YLIM)
axes[-1].axis("off")
axes[0].set_ylabel(CONDITION_YLABEL)
axes[3].set_ylabel(CONDITION_YLABEL)
axes[0].legend(loc=CONDITION_LEGEND_LOC, frameon=False)
fig.suptitle(CONDITION_TITLE, y=1.02, fontsize=9)
save_figure(fig, CONDITION_SAVE_STEM)
plt.show()

## 5. Killer Plot: Global Charge Cut

This is the direct test of the global-imbalance explanation: evaluate
$$\left\langle m_s\right\rangle_{|\Delta Q_s|\leq q_{\rm cut}}$$
as a function of the charge cut. If global charge is the dominant cause, the curve should collapse toward $1/3$ as $q_{\rm cut}$ is tightened.

In [ ]:
QCUT_GRID = np.r_[0.0, np.linspace(0.1, 3.0, 30), np.linspace(3.5, 10.0, 14)]
selected_cycle_rows = []
qcut_rows = []
for ny, sub_ny in features.groupby("Ny"):
    available = np.sort(sub_ny["cycle_label"].unique())
    wanted = sorted(set([int(available[-1]), int(ny), int(max(available[0], 20))]))
    for cycle in wanted:
        sub = sub_ny[sub_ny["cycle_label"] == cycle]
        if sub.empty:
            continue
        selected_cycle_rows.append({"Ny": int(ny), "cycle_label": int(cycle), "n": int(len(sub))})
        for qcut in QCUT_GRID:
            kept = sub[sub["abs_q_pct"] <= qcut]
            qcut_rows.append({
                "Ny": int(ny),
                "cycle_label": int(cycle),
                "q_cut_pct": float(qcut),
                "n": int(len(kept)),
                "mean_slope": float(kept["entropy_slope"].mean()) if len(kept) else np.nan,
                "sem_slope": sem(kept["entropy_slope"]) if len(kept) else np.nan,
                "mean_delta": float(kept["delta_slope"].mean()) if len(kept) else np.nan,
            })
qcut_table = pd.DataFrame(qcut_rows)
selected_cycles = pd.DataFrame(selected_cycle_rows)
save_table(qcut_table, "killer_qcut_slopes.csv")
save_table(selected_cycles, "killer_qcut_selected_cycles.csv")
display(selected_cycles)

In [ ]:
# Plot controls
QCUT_TITLE = r"Global charge cut does not by itself remove the slope excess"
QCUT_XLABEL = r"$q_{\rm cut}$ (%)"
QCUT_YLABEL = r"$\langle m_s\rangle_{|\Delta Q|\leq q_{\rm cut}}$"
QCUT_XLIM = (0.0, 5.0)
QCUT_YLIM = None
QCUT_MIN_N = 3
QCUT_LEGEND_LOC = "best"
QCUT_SAVE_STEM = "killer_qcut_slope_vs_qcut"

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 0.78 * FIG_WIDTH))
for ny in GEOMETRIES_NY:
    final_cycle = int(final_cycle_by_ny[ny])
    sub = qcut_table[(qcut_table["Ny"] == ny) & (qcut_table["cycle_label"] == final_cycle) & (qcut_table["n"] >= QCUT_MIN_N)]
    ax.errorbar(
        sub["q_cut_pct"], sub["mean_slope"], yerr=sub["sem_slope"],
        color=NY_COLORS[ny], marker="o", ms=2.8, lw=1.1, capsize=2,
        label=f"Ny={ny}, cycle={final_cycle}",
    )
ax.axhline(TARGET_SLOPE, color="0.15", ls="--", lw=0.9, label=r"$1/3$")
ax.set_xlabel(QCUT_XLABEL)
ax.set_ylabel(QCUT_YLABEL)
ax.set_xlim(*QCUT_XLIM)
if QCUT_YLIM is not None:
    ax.set_ylim(*QCUT_YLIM)
ax.legend(loc=QCUT_LEGEND_LOC, frameon=False)
ax.set_title(QCUT_TITLE)
save_figure(fig, QCUT_SAVE_STEM)
plt.show()

## 6. Test 3: Cycle Convergence

These plots track the ensemble-mean slope excess and defect observables versus cycle. The fits are deliberately exploratory diagnostics, not asymptotic finite-size claims.

In [ ]:
cycle_summary = (
    features.groupby(["Ny", "cycle_label"])
    .agg(
        n=("entropy_slope", "size"),
        mean_slope=("entropy_slope", "mean"),
        sem_slope=("entropy_slope", sem),
        mean_delta=("delta_slope", "mean"),
        sem_delta=("delta_slope", sem),
        mean_abs_q_pct=("abs_q_pct", "mean"),
        mean_wall_charge_rms=("wall_charge_rms", "mean"),
        mean_wall_frustration=("wall_frustration_mean", "mean"),
        mean_chern_error=("chern_error", "mean"),
        mean_frob_delta=("frob_successive_delta", "mean"),
    )
    .reset_index()
)

def exp_model(c, a, b, tau):
    c0 = np.min(c)
    return a + b * np.exp(-(c - c0) / tau)

def power_model(c, a, b, beta):
    c0 = np.min(c)
    x = (c - c0 + 1.0)
    return a + b * x ** (-beta)

fit_rows = []
for ny, sub in cycle_summary.groupby("Ny"):
    x = sub["cycle_label"].to_numpy(dtype=float)
    y = sub["mean_delta"].to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    xfit = x[mask]
    yfit = y[mask]
    for model_name, model, p0, bounds in [
        ("constant_plus_exponential", exp_model, [float(yfit[-1]), float(yfit[0] - yfit[-1]), max(2.0, ny / 4)], ([-np.inf, -np.inf, 1e-3], [np.inf, np.inf, 1e4])),
        ("constant_plus_power", power_model, [float(yfit[-1]), float(yfit[0] - yfit[-1]), 1.0], ([-np.inf, -np.inf, 1e-3], [np.inf, np.inf, 10.0])),
    ]:
        try:
            popt, pcov = curve_fit(model, xfit, yfit, p0=p0, bounds=bounds, maxfev=20000)
            residual = yfit - model(xfit, *popt)
            rmse = float(np.sqrt(np.mean(residual * residual)))
            status = "ok"
        except Exception as exc:
            popt = [np.nan, np.nan, np.nan]
            rmse = np.nan
            status = f"failed: {exc}"
        fit_rows.append({
            "Ny": int(ny),
            "model": model_name,
            "plateau_delta": float(popt[0]),
            "amplitude": float(popt[1]),
            "scale_or_beta": float(popt[2]),
            "rmse": rmse,
            "status": status,
        })
cycle_fits = pd.DataFrame(fit_rows)
save_table(cycle_summary, "test3_cycle_convergence_summary.csv")
save_table(cycle_fits, "test3_cycle_convergence_fits.csv")
display(cycle_fits)

In [ ]:
# Plot controls
CONV_TITLE = r"Cycle convergence of the entropy-slope excess"
CONV_XLABEL = "cycle"
CONV_YLABEL = r"$\overline{m_s}-1/3$"
CONV_YLIM = None
CONV_LEGEND_LOC = "best"
CONV_SAVE_STEM = "test3_delta_slope_vs_cycle"
SHOW_FIT_OVERLAYS = True

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 0.78 * FIG_WIDTH))
for ny, sub in cycle_summary.groupby("Ny"):
    ax.errorbar(
        sub["cycle_label"], sub["mean_delta"], yerr=sub["sem_delta"],
        color=NY_COLORS[int(ny)], lw=1.1, capsize=1.8, label=f"Ny={ny}",
    )
    if SHOW_FIT_OVERLAYS:
        fit = cycle_fits[(cycle_fits["Ny"] == ny) & (cycle_fits["model"] == "constant_plus_exponential")]
        if len(fit) and fit.iloc[0]["status"] == "ok":
            row = fit.iloc[0]
            xgrid = np.linspace(sub["cycle_label"].min(), sub["cycle_label"].max(), 200)
            ygrid = exp_model(xgrid, row["plateau_delta"], row["amplitude"], row["scale_or_beta"])
            ax.plot(xgrid, ygrid, color=NY_COLORS[int(ny)], lw=0.8, ls="--", alpha=0.75)
ax.axhline(0.0, color="0.15", lw=0.8, ls=":")
ax.set_xlabel(CONV_XLABEL)
ax.set_ylabel(CONV_YLABEL)
if CONV_YLIM is not None:
    ax.set_ylim(*CONV_YLIM)
ax.legend(loc=CONV_LEGEND_LOC, frameon=False)
ax.set_title(CONV_TITLE)
save_figure(fig, CONV_SAVE_STEM)
plt.show()

In [ ]:
# Plot controls
DEFECT_TITLE = r"Mean defect diagnostics versus cycle"
DEFECT_XLABEL = "cycle"
DEFECT_YLABEL = "normalized mean metric"
DEFECT_METRICS = ["mean_abs_q_pct", "mean_wall_charge_rms", "mean_wall_frustration", "mean_chern_error", "mean_frob_delta"]
DEFECT_LABELS = {
    "mean_abs_q_pct": r"$|\Delta Q|$",
    "mean_wall_charge_rms": "wall charge",
    "mean_wall_frustration": "wall frustration",
    "mean_chern_error": r"$|C-1|$",
    "mean_frob_delta": r"$\Delta G$",
}
DEFECT_LEGEND_LOC = "upper right"
DEFECT_SAVE_STEM = "test3_defect_metrics_vs_cycle"

fig, axes = plt.subplots(1, 3, figsize=(2.1 * FIG_WIDTH, 0.72 * FIG_WIDTH), sharey=True)
for ax, (ny, sub) in zip(axes, cycle_summary.groupby("Ny")):
    for metric in DEFECT_METRICS:
        y = sub[metric].to_numpy(dtype=float)
        ymin, ymax = np.nanmin(y), np.nanmax(y)
        yn = (y - ymin) / (ymax - ymin) if ymax > ymin else np.zeros_like(y)
        ax.plot(sub["cycle_label"], yn, lw=1.0, label=DEFECT_LABELS.get(metric, metric))
    ax.set_title(f"Ny={ny}")
    ax.set_xlabel(DEFECT_XLABEL)
axes[0].set_ylabel(DEFECT_YLABEL)
axes[-1].legend(loc=DEFECT_LEGEND_LOC, frameon=False)
fig.suptitle(DEFECT_TITLE, y=1.04, fontsize=9)
save_figure(fig, DEFECT_SAVE_STEM)
plt.show()

## 7. Test 4: Averaging Order

Compare the trajectory-averaged slope $\overline{m_s}$ against the slope obtained after first averaging the entropy curve,
$$m[\overline{S_s(A_y)}].$$
This is a nonlinear-observable diagnostic; it does not require averaging covariance matrices.

In [ ]:
avg_rows = []
for ny in GEOMETRIES_NY:
    ent_npz = RUN_DATA[ny]["entropy_npz"]
    entropy = ent_npz["entropy_y0avg_vs_ay"].astype(np.float64, copy=False)
    ay_values = ent_npz["ay_values"].astype(int)
    cycles = ent_npz["cycle_labels"].astype(int)
    samples = ent_npz["sample_indices"].astype(int)
    scatter_ny = features[features["Ny"] == ny]
    for ci, cycle in enumerate(cycles):
        if cycle < scatter_ny["cycle_label"].min():
            continue
        curve_mean = np.nanmean(entropy[:, ci, :], axis=0)
        slope_curve, intercept_curve, r2_curve, nfit = fit_entropy_curve(ay_values, curve_mean, ny, FIT_AY_MIN)
        sub = scatter_ny[scatter_ny["cycle_label"] == cycle]
        avg_rows.append({
            "Ny": int(ny),
            "cycle_label": int(cycle),
            "n_samples_curve": int(len(samples)),
            "mean_of_sample_slopes": float(sub["entropy_slope"].mean()) if len(sub) else np.nan,
            "sem_of_sample_slopes": sem(sub["entropy_slope"]) if len(sub) else np.nan,
            "slope_of_mean_entropy_curve": float(slope_curve),
            "curve_fit_r2": float(r2_curve),
            "n_fit_points": int(nfit),
            "difference_mean_slope_minus_slope_of_mean_curve": float(sub["entropy_slope"].mean() - slope_curve) if len(sub) else np.nan,
        })
avg_order = pd.DataFrame(avg_rows)
save_table(avg_order, "test4_averaging_order.csv")
display(avg_order.groupby("Ny").tail(1))

In [ ]:
# Plot controls
AVG_TITLE = r"Averaging-order diagnostic: $\overline{m_s}$ versus $m[\overline{S(A_y)}]$"
AVG_XLABEL = "cycle"
AVG_YLABEL = r"entropy slope"
AVG_YLIM = None
AVG_LEGEND_LOC = "best"
AVG_SAVE_STEM = "test4_averaging_order"

fig, axes = plt.subplots(1, 3, figsize=(2.1 * FIG_WIDTH, 0.72 * FIG_WIDTH), sharey=True)
for ax, (ny, sub) in zip(axes, avg_order.groupby("Ny")):
    ax.plot(sub["cycle_label"], sub["mean_of_sample_slopes"], color=NY_COLORS[int(ny)], lw=1.1, label=r"$\overline{m_s}$")
    ax.plot(sub["cycle_label"], sub["slope_of_mean_entropy_curve"], color="0.15", lw=1.0, ls="--", label=r"$m[\overline{S}]$")
    ax.axhline(TARGET_SLOPE, color="0.35", lw=0.8, ls=":")
    ax.set_title(f"Ny={ny}")
    ax.set_xlabel(AVG_XLABEL)
    if AVG_YLIM is not None:
        ax.set_ylim(*AVG_YLIM)
axes[0].set_ylabel(AVG_YLABEL)
axes[0].legend(loc=AVG_LEGEND_LOC, frameon=False)
fig.suptitle(AVG_TITLE, y=1.04, fontsize=9)
save_figure(fig, AVG_SAVE_STEM)
plt.show()

## 8. Test 5: Charge-Neutralized and Defect-Neutralized Ensembles

This combines the preceding ideas into selectors: low global charge, low wall charge, low wall frustration, low Chern error, and a combined standardized defect score.

In [ ]:
selector_metrics = {
    "global_charge": ["abs_q_pct"],
    "wall_charge": ["wall_charge_rms"],
    "wall_frustration": ["wall_frustration_mean"],
    "chern_error": ["chern_error"],
    "combined_wall_defect": ["wall_charge_rms", "wall_frustration_mean", "chern_error"],
    "combined_all": ["abs_q_pct", "wall_charge_rms", "wall_frustration_mean", "chern_error", "frob_successive_delta"],
}
neutral_rows = []
for ny, sub_ny in final_features.groupby("Ny"):
    for selector, metrics in selector_metrics.items():
        score_parts = []
        for metric in metrics:
            v = sub_ny[metric].to_numpy(dtype=float)
            sigma = np.nanstd(v)
            z = (v - np.nanmean(v)) / sigma if sigma > 0 else np.zeros_like(v)
            score_parts.append(z)
        score = np.nanmean(np.vstack(score_parts), axis=0)
        ranked = sub_ny.assign(selector_score=score).sort_values("selector_score", kind="mergesort")
        base_excess = float(sub_ny["delta_slope"].mean())
        for frac in RETAINED_FRACTIONS:
            n_keep = max(1, int(math.ceil(frac * len(ranked))))
            kept = ranked.head(n_keep)
            neutral_rows.append({
                "Ny": int(ny),
                "selector": selector,
                "retained_fraction": float(frac),
                "n": int(n_keep),
                "mean_slope": float(kept["entropy_slope"].mean()),
                "sem_slope": sem(kept["entropy_slope"]),
                "mean_delta": float(kept["delta_slope"].mean()),
                "excess_reduction_fraction": float(1.0 - kept["delta_slope"].mean() / base_excess) if base_excess != 0 else np.nan,
            })
neutralized = pd.DataFrame(neutral_rows)
save_table(neutralized, "test5_neutralized_ensemble_comparison.csv")
display(neutralized.head(18))

In [ ]:
# Plot controls
NEUT_TITLE = r"Which neutralization most reduces $m_s-1/3$?"
NEUT_XLABEL = "retained fraction"
NEUT_YLABEL = "excess reduction fraction"
NEUT_YLIM = None
NEUT_LEGEND_LOC = "best"
NEUT_SAVE_STEM = "test5_neutralized_ensemble_comparison"
NEUT_SELECTORS = ["global_charge", "wall_charge", "wall_frustration", "chern_error", "combined_wall_defect", "combined_all"]

fig, axes = plt.subplots(1, 3, figsize=(2.1 * FIG_WIDTH, 0.72 * FIG_WIDTH), sharey=True)
for ax, (ny, sub_ny) in zip(axes, neutralized.groupby("Ny")):
    for selector in NEUT_SELECTORS:
        sub = sub_ny[sub_ny["selector"] == selector]
        ax.plot(sub["retained_fraction"], sub["excess_reduction_fraction"], marker="o", ms=2.8, lw=1.0, label=selector)
    ax.axhline(0.0, color="0.2", lw=0.8, ls=":")
    ax.set_title(f"Ny={ny}")
    ax.set_xlabel(NEUT_XLABEL)
    if NEUT_YLIM is not None:
        ax.set_ylim(*NEUT_YLIM)
axes[0].set_ylabel(NEUT_YLABEL)
axes[-1].legend(loc=NEUT_LEGEND_LOC, frameon=False)
fig.suptitle(NEUT_TITLE, y=1.04, fontsize=9)
save_figure(fig, NEUT_SAVE_STEM)
plt.show()

## 9. Test 6: Optional X-Window Localized Contours from Raw N16 Covariances

This section is disabled by default. When enabled, it loads a capped number of pure-state covariance snapshots and computes strip entropies using the same occupation-spectrum formula as the streaming observer, then compares full-width, left-wall, right-wall, combined-wall, and bulk-only window slopes.

Because this recomputes many eigenspectra from large matrices, start with the default sample cap before increasing it.

In [ ]:
def raw_snapshot_paths(raw_ny: int = RAW_NY, nshell: int = RAW_NSHELL):
    raw_root = ROOT / "colab_small_system_testing/gpu_data/pure_state_covariance_snapshots/runs"
    run_root = raw_root / f"N16x{raw_ny}_nsh{nshell}_perfect_correction"
    latest_path = run_root / "latest_run.json"
    if not latest_path.exists():
        raise FileNotFoundError(latest_path)
    with latest_path.open("r", encoding="utf-8") as fh:
        latest = json.load(fh)
    run_id = latest.get("run_id") or latest.get("latest_run_id")
    if run_id is None:
        candidates = sorted(run_root.glob("run_*/batch_*_snapshots.npy"))
    else:
        candidates = sorted((run_root / str(run_id)).glob("batch_*_snapshots.npy"))
    ensure_no_forbidden_paths([latest_path, *candidates])
    return latest, candidates

def mode_indices_window(nx: int, ny: int, ay: int, y0: int, x_mask: np.ndarray) -> np.ndarray:
    x_values = np.flatnonzero(np.asarray(x_mask, dtype=bool))
    idx = []
    for dy in range(int(ay)):
        y = (int(y0) + dy) % int(ny)
        for x in x_values:
            idx.extend([2 * int(x) + 2 * int(nx) * y, 2 * int(x) + 2 * int(nx) * y + 1])
    return np.asarray(idx, dtype=np.int64)

def entropy_from_restricted_covariance(sub_g: np.ndarray, eps: float = 1e-12) -> float:
    if sub_g.size == 0:
        return 0.0
    eye = np.eye(sub_g.shape[0], dtype=sub_g.dtype)
    occ = 0.5 * (sub_g + eye)
    occ = 0.5 * (occ + occ.conj().T)
    vals = np.linalg.eigvalsh(occ).real
    vals = np.clip(vals, eps, 1.0 - eps)
    return float(-np.sum(vals * np.log(vals) + (1.0 - vals) * np.log(1.0 - vals)))

def compute_raw_xwindow_slopes(raw_ny: int = RAW_NY, nshell: int = RAW_NSHELL, sample_limit: int = RAW_SAMPLE_LIMIT):
    latest, paths = raw_snapshot_paths(raw_ny, nshell)
    if not paths:
        raise FileNotFoundError("No raw snapshot batches found")
    snapshots = np.load(paths[0], mmap_mode="r")
    sample_count, snapshot_count, dim, _ = snapshots.shape
    nx = int(latest.get("Nx", latest.get("nx", 16)))
    ny = int(latest.get("Ny", latest.get("ny", raw_ny)))
    cycles = latest.get("snapshot_cycles", list(range(snapshot_count)))
    final_snapshot_index = snapshot_count - 1
    sample_stop = min(int(sample_limit), int(sample_count))
    ay_values = np.arange(max(2, FIT_AY_MIN), ny // 2 + 1, dtype=int)
    y0_values = np.arange(0, ny, max(1, int(RAW_Y0_STRIDE)), dtype=int)
    xL, xR, wall_mask, bulk_mask = domain_wall_masks(nx, WALL_RADIUS)
    left_mask = np.zeros(nx, dtype=bool); left_mask[max(0, xL-WALL_RADIUS):min(nx, xL+WALL_RADIUS+1)] = True
    right_mask = np.zeros(nx, dtype=bool); right_mask[max(0, xR-WALL_RADIUS):min(nx, xR+WALL_RADIUS+1)] = True
    windows = {
        "full_width": np.ones(nx, dtype=bool),
        "left_wall": left_mask,
        "right_wall": right_mask,
        "both_walls": wall_mask,
        "bulk_only": bulk_mask,
    }
    rows = []
    for sample in range(sample_stop):
        G = np.asarray(snapshots[sample, final_snapshot_index])
        for window_name, x_mask in windows.items():
            curve = []
            for ay in ay_values:
                vals = []
                for y0 in y0_values:
                    idx = mode_indices_window(nx, ny, int(ay), int(y0), x_mask)
                    sub_g = G[np.ix_(idx, idx)]
                    vals.append(entropy_from_restricted_covariance(sub_g))
                curve.append(float(np.mean(vals)))
            slope, intercept, r2, nfit = fit_entropy_curve(ay_values, np.asarray(curve), ny, FIT_AY_MIN)
            rows.append({
                "Ny": int(ny),
                "Nx": int(nx),
                "nshell": int(nshell),
                "sample_index": int(sample),
                "snapshot_cycle": int(cycles[final_snapshot_index]) if final_snapshot_index < len(cycles) else int(final_snapshot_index),
                "window": window_name,
                "entropy_slope": float(slope),
                "intercept": float(intercept),
                "r2": float(r2),
                "n_fit_points": int(nfit),
                "n_y0": int(len(y0_values)),
                "n_x": int(np.count_nonzero(x_mask)),
            })
    return pd.DataFrame(rows)

if RUN_RAW_COVARIANCE_SECTION:
    raw_xwindow = compute_raw_xwindow_slopes()
    save_table(raw_xwindow, "test6_raw_xwindow_slopes.csv")
    display(raw_xwindow)
else:
    raw_xwindow = pd.DataFrame()
    print("Skipped raw-covariance x-window section. Set RUN_RAW_COVARIANCE_SECTION=True to execute it.")

In [ ]:
# Plot controls
RAW_TITLE = r"Optional raw N16 x-window contour slopes"
RAW_XLABEL = "x window"
RAW_YLABEL = r"$m_s$"
RAW_YLIM = None
RAW_LEGEND_LOC = "best"
RAW_SAVE_STEM = "test6_raw_xwindow_slopes"
RAW_WINDOW_ORDER = ["full_width", "left_wall", "right_wall", "both_walls", "bulk_only"]

if RUN_RAW_COVARIANCE_SECTION and not raw_xwindow.empty:
    raw_summary = raw_xwindow.groupby("window").agg(
        mean_slope=("entropy_slope", "mean"),
        sem_slope=("entropy_slope", sem),
        n=("entropy_slope", "size"),
    ).reindex(RAW_WINDOW_ORDER).reset_index()
    save_table(raw_summary, "test6_raw_xwindow_slope_summary.csv")
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, 0.78 * FIG_WIDTH))
    x = np.arange(len(raw_summary))
    ax.errorbar(x, raw_summary["mean_slope"], yerr=raw_summary["sem_slope"], fmt="o", capsize=2, color="0.1")
    ax.axhline(TARGET_SLOPE, color="0.25", ls="--", lw=0.8, label=r"full target $1/3$")
    ax.axhline(TARGET_SLOPE / 2.0, color="0.5", ls=":", lw=0.8, label=r"single-wall target $1/6$")
    ax.set_xticks(x)
    ax.set_xticklabels(raw_summary["window"], rotation=30, ha="right")
    ax.set_xlabel(RAW_XLABEL)
    ax.set_ylabel(RAW_YLABEL)
    if RAW_YLIM is not None:
        ax.set_ylim(*RAW_YLIM)
    ax.legend(loc=RAW_LEGEND_LOC, frameon=False)
    ax.set_title(RAW_TITLE)
    save_figure(fig, RAW_SAVE_STEM)
    plt.show()
else:
    print("No raw-covariance figure emitted because RUN_RAW_COVARIANCE_SECTION=False or the raw table is empty.")

## 10. Summary Cell

The decision table below applies simple, transparent criteria to the tables above. It is meant to triage explanations for the excess slope, not to close the finite-size scaling question.

In [ ]:
def verdict(flag, yes="supported", no="not supported"):
    return yes if bool(flag) else no

summary_rows = []
# Global charge criterion: low charge cut should remove at least half the final-cycle excess or move within 0.003 of 1/3.
global_by_ny = []
for ny in GEOMETRIES_NY:
    final_cycle = final_cycle_by_ny[ny]
    base = float(final_features[final_features["Ny"] == ny]["delta_slope"].mean())
    sub = qcut_table[(qcut_table["Ny"] == ny) & (qcut_table["cycle_label"] == final_cycle) & (qcut_table["n"] >= 3)]
    low = sub[sub["q_cut_pct"] <= 0.5]
    best_abs = float(low["mean_delta"].abs().min()) if len(low) else np.nan
    best_reduction = float(1.0 - low["mean_delta"].min() / base) if len(low) and base != 0 else np.nan
    global_by_ny.append((best_abs <= 0.003) or (best_reduction >= 0.5))
summary_rows.append({
    "hypothesis": "global charge imbalance",
    "verdict": verdict(any(global_by_ny)),
    "criterion": "q_cut <= 0.5% moves mean slope within 0.003 of 1/3 or removes >=50% of excess for any Ny",
    "diagnostic_file": "tables/killer_qcut_slopes.csv",
})

# Local-wall criterion: wall selectors/correlations reduce the excess and correlate with it.
wall_cond = conditioned[(conditioned["metric"].isin(["wall_charge_rms", "wall_frustration_mean"])) & (conditioned["retained_fraction"] == 0.2)]
wall_reduction = float(wall_cond["excess_reduction_fraction"].max()) if len(wall_cond) else np.nan
wall_corr = correlations[correlations["metric"].isin(["wall_charge_rms", "wall_frustration_mean"])]
wall_corr_strength = float(wall_corr["spearman_r"].abs().max()) if len(wall_corr) else np.nan
wall_reduction_geometries = int(wall_cond[wall_cond["excess_reduction_fraction"] >= 0.30]["Ny"].nunique()) if len(wall_cond) else 0
wall_supported = (wall_reduction >= 0.30) and (wall_corr_strength >= 0.25)
wall_mixed = (wall_reduction >= 0.30) or (wall_corr_strength >= 0.25)
summary_rows.append({
    "hypothesis": "local wall charge/defect structure",
    "verdict": "supported" if wall_supported else ("mixed support" if wall_mixed else "not supported"),
    "criterion": "lowest 20% wall metrics remove >=30% excess in at least one geometry and any wall-metric |Spearman rho| >= 0.25",
    "diagnostic_file": "tables/test2_conditioned_subset_slopes.csv; tables/test1_metric_correlations.csv",
})

# Convergence criterion: last-half slope excess trends downward or fitted plateau is closer to zero than the final point.
conv_flags = []
for ny, sub in cycle_summary.groupby("Ny"):
    tail = sub[sub["cycle_label"] >= sub["cycle_label"].median()]
    trend = np.polyfit(tail["cycle_label"], tail["mean_delta"], 1)[0] if len(tail) >= 3 else np.nan
    final_delta = float(tail.iloc[-1]["mean_delta"])
    fit = cycle_fits[(cycle_fits["Ny"] == ny) & (cycle_fits["model"] == "constant_plus_exponential")]
    plateau = float(fit.iloc[0]["plateau_delta"]) if len(fit) and fit.iloc[0]["status"] == "ok" else np.nan
    conv_flags.append((np.isfinite(trend) and trend < -1e-4) or (np.isfinite(plateau) and abs(plateau) < 0.7 * abs(final_delta)))
summary_rows.append({
    "hypothesis": "incomplete cycle convergence",
    "verdict": verdict(any(conv_flags)),
    "criterion": "last-half trend is decreasing or exponential plateau is substantially closer to target",
    "diagnostic_file": "tables/test3_cycle_convergence_summary.csv; tables/test3_cycle_convergence_fits.csv",
})

# Averaging-order criterion.
avg_final = avg_order.groupby("Ny").tail(1).copy()
max_avg_order_gap = float(avg_final["difference_mean_slope_minus_slope_of_mean_curve"].abs().max())
summary_rows.append({
    "hypothesis": "entropy-curve averaging order",
    "verdict": verdict(max_avg_order_gap > 0.003),
    "criterion": "max final-cycle |mean(sample slopes)-slope(mean entropy curve)| > 0.003",
    "diagnostic_file": "tables/test4_averaging_order.csv",
})

if RUN_RAW_COVARIANCE_SECTION and not raw_xwindow.empty:
    raw_summary = raw_xwindow.groupby("window")["entropy_slope"].mean()
    full_ok = abs(float(raw_summary.get("full_width", np.nan)) - TARGET_SLOPE) < 0.02
    wall_ok = min(
        abs(float(raw_summary.get("left_wall", np.nan)) - TARGET_SLOPE / 2.0),
        abs(float(raw_summary.get("right_wall", np.nan)) - TARGET_SLOPE / 2.0),
    ) < 0.02
    raw_verdict = verdict(full_ok and wall_ok)
    criterion = "full width near 1/3 and at least one single wall near 1/6 within 0.02"
else:
    raw_verdict = "not evaluated"
    criterion = "raw covariance section skipped by RUN_RAW_COVARIANCE_SECTION=False"
summary_rows.append({
    "hypothesis": "x-window contour contamination",
    "verdict": raw_verdict,
    "criterion": criterion,
    "diagnostic_file": "tables/test6_raw_xwindow_slopes.csv when enabled",
})

decision_table = pd.DataFrame(summary_rows)
save_table(decision_table, "summary_decision_table.csv")
display(decision_table)

print("Key numerical handles:")
print(f"  max final averaging-order gap: {max_avg_order_gap:.6f}")
print(f"  best low-wall-metric excess reduction at retained fraction 0.2: {wall_reduction:.3f}")
print(f"  pooled wall metric |Spearman rho| max: {wall_corr_strength:.3f}")
print(f"  output root: {OUTPUT_ROOT}")